In [ ]:
from openai import OpenAI
from opensearchpy import OpenSearch
import torch
from sentence_transformers import SentenceTransformer 
import json
import time
from tqdm import tqdm
import pandas as pd
import os
import random
from collections import Counter

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = SentenceTransformer("BAAI/bge-m3", device=device)

In [ ]:
df_fora = pd.read_csv("/home/jjunqueira/questionanswer/julia/documentos_fora_regis.csv")

In [ ]:
df_reescrito_regis = pd.read_csv("/home/jjunqueira/questionanswer/julia/parciais/regis-correcao.csv")

In [ ]:
questions_list = {
    'question_regis': df_reescrito_regis['question_regis'].dropna().tolist()
}


print(questions_list)

In [ ]:
client = OpenSearch(
    hosts=[{'host': 'seu-host.com.br', 'port': 8000}],
    http_auth=('xxxxx', 'xxxxx'),
    use_ssl=True,
    verify_certs=False, 
    timeout=30)

In [ ]:
def run_query(query):
    query = 'query: ' + query

    emb_qry = model.encode(query, show_progress_bar=False)
    emb_qry = emb_qry.tolist()

    query_body = {"size": 20,
        "query": {"knn": {"embedding": {"vector": emb_qry, "k": 10}}},
        "_source": False,
            "fields": ["docid", "json_name", "text"],
    }
    
    response = client.search(
        body = query_body,
        index = 'regis3'
    )
    chunks = []
    for j, hit in enumerate(response["hits"]["hits"]):
        chunks.append({
            'index': hit['_index'],
            'order': j,
            'id': hit['_id'],
            'score': hit['_score'],
            'chunk': hit['fields']['text'][0]
        })
    return chunks

In [ ]:
tst = run_query("Qual tipo de ambiente deposicional favorece a deposição e cimentação da micrita, considerando os processos descritos para sua formação?")

In [ ]:
print(tst)

In [ ]:
print(questions_list['question_regis'])

In [ ]:
questions_regis = {}

for question in questions_list['question_regis']:
    chunks = run_query(question)
    questions_regis[question] = chunks  # cada chave é uma pergunta, e o valor é a lista de 20 chunks

with open('chunks_por_pergunta.json', 'w', encoding='utf-8') as f:
    json.dump(questions_regis, f, ensure_ascii=False, indent=2)

====================================================

In [ ]:
# código para similaridade entre chunks regis/chunks fora

chunks_fora = df_fora["chunk_1"].tolist()  # ajusta se a coluna tiver outro nome

resultados = []

for chunk_fora in tqdm(chunks_fora):
    query = "query: " + chunk_fora
    emb_qry = model.encode(query, show_progress_bar=False).tolist()

    query_body = query_body = {
        "size": 20,
        "query": {
            "knn": {
                "embedding": {
                    "vector": emb_qry,
                    "k": 20
                }
            }
            
        },
        "_source": False,
        "fields": ["docid", "json_name", "text"],
    }
    response = client.search(body=query_body, index="regis3")
    
    similar_chunks = []
    for hit in response["hits"]["hits"]:
        similar_chunks.append({
            "score": hit["_score"],
            "chunk_text": hit["fields"].get("text", "")[0],
            "id": hit["_id"],
            "json_name": hit["fields"].get("json_name", ""),
        })

    resultados.append({
        "chunk_fora": chunk_fora,
        "mais_similares": similar_chunks
    })

with open("resultados_similares.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)




In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

with open("resultados_similares.json", "r", encoding="utf-8") as f:
    resultados = json.load(f)

novos_resultados = []

for entrada in tqdm(resultados):
    chunk_fora = entrada["chunk_fora"]
    similares = entrada["mais_similares"]

    
    emb_fora = model.encode(chunk_fora, convert_to_numpy=True)

    textos_similares = [s["chunk_text"] for s in similares]
    emb_similares = model.encode(textos_similares, convert_to_numpy=True)

    sims = cosine_similarity([emb_fora], emb_similares)[0]

    novos_similares = []
    for s, score in zip(similares, sims):
        novos_similares.append({
            "chunk_text": s["chunk_text"],
            "opensearch_cosine": s["score"],
            "sklearn_cosine": float(score),
            "id": s["id"],
            "json_name": s.get("json_name", "")
        })

    novos_resultados.append({
        "chunk_fora": chunk_fora,
        "mais_similares": novos_similares
    })

with open("comparacao_chunks_similares.json", "w", encoding="utf-8") as f:
    json.dump(novos_resultados, f, ensure_ascii=False, indent=2)

In [ ]:
print(tst)

In [ ]:
import csv

caminho_json_entrada = "comparacao_chunks_similares.json"
caminho_csv_saida = "resultado_similaridade.csv"


LIMITE_SCORE = 0.70


try:
    
    with open(caminho_json_entrada, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    print(f"Arquivo JSON '{caminho_json_entrada}' carregado com sucesso.")

except FileNotFoundError:
    print(f"ERRO: Arquivo de entrada não encontrado em: {caminho_json_entrada}")
    exit() 
except json.JSONDecodeError:
    print(f"ERRO: O arquivo '{caminho_json_entrada}' não contém um JSON válido.")
    exit() # encerra se o JSON for inválido


with open(caminho_csv_saida, 'w', newline='', encoding='utf-8-sig') as arquivo_csv:
    
    escritor_csv = csv.writer(arquivo_csv)

    cabecalho = ["chunk_fora", "chunk_similar", "score_cosine", "id_similar"]
    escritor_csv.writerow(cabecalho)

    for item in dados:
        chunk_fora = item.get("chunk_fora", "")

        # reseta as variáveis para o chunk similar a cada iteração
        texto_similar = ""
        score_similar = ""
        id_similar = "" 
        # pega a lista de similares, se existir
        lista_similares = item.get("mais_similares", [])

        if lista_similares:
            primeiro_similar = lista_similares[0]
            score = primeiro_similar.get("opensearch_cosine", 0.0)

            # aplica a condição do score
            if score >= LIMITE_SCORE:
                texto_similar = primeiro_similar.get("chunk_text", "")
                score_similar = score
                id_similar = primeiro_similar.get("id", "")

        linha_para_salvar = [chunk_fora, texto_similar, score_similar, id_similar]

        escritor_csv.writerow(linha_para_salvar)

print(f"\nProcesso concluído! Arquivo '{caminho_csv_saida}' salvo com sucesso.")

In [ ]:
def generate_llm_response(prompt, model):

    client = OpenAI(
        api_key='sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX',
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"user","content": prompt},
        ]
    )
    return completion.choices[0].message.content

def make_prompt(prompt_file, completion, model='gpt-4o'):
    with open(prompt_file) as pf:
        pre_prompt = pf.read()
        return generate_llm_response(pre_prompt+completion,model)

In [ ]:
def try_to_decode_json(str):
    try:
        ans = json.loads(str)
    except:
        ans = json.loads(str[8:-4])
    return ans

In [ ]:
import math

def is_random(s: str) -> bool:
    """
    Classifica uma string com base em algumas heurísticas:
      - Proporção de caracteres alfabéticos.
      - Proporção de espaços (para indicar a presença de palavras).
      - Validação das palavras encontradas (se possuem pelo menos uma vogal).
      - Entropia da string (para medir o grau de aleatoriedade).
    
    Retorna um dicionário com os valores calculados e a decisão final.
    A decisão final pode ser "texto" ou "aleatório" baseada em thresholds definidos.
    """
    resultado = {}
    total = len(s)
    
    if total == 0:
        resultado['final'] = "aleatório"  # ou pode tratar caso especial de string vazia
        return resultado

    # Heurística 1: Proporção de caracteres alfabéticos
    count_alpha = sum(1 for c in s if c.isalpha())
    ratio_alpha = count_alpha / total
    resultado['ratio_alpha'] = ratio_alpha

    # Heurística 2: Proporção de espaços (indicativo de separação de palavras)
    count_spaces = s.count(" ")
    ratio_spaces = count_spaces / total
    resultado['ratio_spaces'] = ratio_spaces

    # Heurística 3: Verificação das palavras válidas (contendo ao menos uma vogal)
    palavras = s.split()
    validas = 0
    vogais = "aeiouAEIOUáéíóúÁÉÍÓÚâêîôûÂÊÎÔÛãõÃÕ"
    for palavra in palavras:
        if any(letra in vogais for letra in palavra):
            validas += 1
    ratio_validas = validas / len(palavras) if palavras else 0
    resultado['ratio_validas'] = ratio_validas

    # Heurística 4: Cálculo da entropia da string (medida de aleatoriedade)
    freq = {}
    for char in s:
        freq[char] = freq.get(char, 0) + 1
    entropia = -sum((count / total) * math.log2(count / total) for count in freq.values())
    resultado['entropia'] = entropia

    # Decisão final: definir thresholds para as heurísticas.
    # Estes thresholds podem ser ajustados conforme o domínio de aplicação.
    # Exemplo:
    # - Se a proporção de letras for alta (ex: > 0.6),
    # - Se a maioria das palavras tiver pelo menos uma vogal (ex: > 0.5),
    # - Se a entropia estiver dentro de um intervalo "típico" para texto (ex: entre 3.0 e 5.5)
    if ratio_alpha > 0.6 and ratio_validas > 0.5 and (3.0 <= entropia <= 5.5):
        resultado['final'] = "texto"
    else:
        resultado['final'] = "aleatório"
    
    return resultado['final'] == "aleatório"


In [ ]:
df_simil = pd.read_csv("/resultado_similaridade.csv")

In [ ]:
import json
import re

def try_to_decode_json(response_str):
    """
    Tenta extrair e decodificar um JSON válido de uma string de resposta do LLM.
    Retorna um dicionário se for bem-sucedido, ou None se falhar.
    """
    if not response_str or not isinstance(response_str, str):
        return None

    # expressão regular para capturar o conteúdo entre chaves
    matches = re.findall(r'\{.*?\}', response_str, re.DOTALL)

    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue  # tenta o próximo match, se houver

    # fallback: tenta carregar diretamente como JSON se não encontrar com regex
    try:
        return json.loads(response_str)
    except json.JSONDecodeError:
        return None


In [ ]:
def is_valid_chunk(text):
    """Verifica se o texto do chunk não é nulo ou vazio."""
    return pd.notna(text) and text.strip() != ""

#esse trecho de código é apenas um teste que foi feito para verificar se o modelo respondia perguntas parcialmente, depois de termos reescrito 
#as perguntas. não é necessário usar esse trecho para a geração final das perguntas parciais,
#mas ele foi importante para validar a abordagem de usar os chunks retornados pela busca semântica.

def gerar_resposta_para_question(question, chunks, prompt_template_file, model_name):
    """
    Gera uma resposta com base em uma pergunta e nos 20 chunks retornados por busca semântica.
    """
    if not is_valid_chunk(question) or not chunks:
        print("Pergunta inválida ou sem chunks.")
        return None

    # junta os chunks em um único contexto
    contexto = "\n\n".join(
        f"CHUNK {i+1}: {chunk['chunk']}" 
        for i, chunk in enumerate(chunks) 
        if is_valid_chunk(chunk.get('chunk', ''))
    )    # prepara o prompt
    prompt_text = f"""CONTEXTO:\n{contexto}\n\nCONSULTA:\n{question}\n\nResponda agora de forma clara, objetiva e indicando os chunks usados como descrito acima."""

    # gera resposta com o modelo
    response_llm = make_prompt(prompt_template_file, prompt_text, model=model_name)
    print(response_llm)
    # tenta decodificar a resposta
    question_data = try_to_decode_json(response_llm)

    if question_data is None:
        print("Erro ao decodificar a resposta.")
        return None

    # retorna dicionário com dados completos
    return {
        'question': question,
        'answer': question_data.get('resposta', 'ERRO'),
        'justificativa': question_data.get('justificativa', 'ERRO'),
        'chunks_utilizados': [chunk['chunk'] for chunk in chunks],
        'ids_chunks': [chunk['id'] for chunk in chunks],
        'scores': [chunk['score'] for chunk in chunks],
    }


In [ ]:
respostas_geradas = []
path = "prompts/prompt_regis.txt"
for question, chunks in questions_regis.items():
    resultado = gerar_resposta_para_question(question, chunks, prompt_template_file=path, model_name='gpt-4')
    if resultado:
        respostas_geradas.append(resultado)

In [ ]:
def is_valid_chunk(text):
    """Verifica se o texto do chunk não é nulo ou vazio."""
    return pd.notna(text) and text.strip() != ""

def gerar_pergunta_da_linha_csv(row, row_index, prompt_template_file, model_name):
    """
    Pega uma linha de um DataFrame, extrai os dois chunks e gera uma pergunta composta.
    """
    chunk_fora = row.chunk_fora
    chunk_similar = row.chunk_similar
    id_similar = row.id_similar
    score_similar = row.score_cosine

    
    if not is_valid_chunk(chunk_similar):
        print(f"Linha {row_index}: Pulando, 'Chunk Mais Similar' está vazio.")
        return None

    # usa os dois chunks da mesma linha para o prompt
    prompt_text = f"\n\nDocumento 1: {{{chunk_similar}}}\n\nDocumento 2: {{{chunk_fora}}}"
    
    # chama a sua função make_prompt, que por sua vez chama a API
    response_llm = make_prompt(prompt_template_file, prompt_text, model=model_name)
    
    # decodifica a resposta JSON do modelo
    question_data = try_to_decode_json(response_llm)
            
    # monta o dicionário de resultado
    id_chunk_fora = f"csv_row_{row_index}"
    return {
        'question': question_data.get('pergunta', 'ERRO'),
        'answer': question_data.get('resposta', 'ERRO'),
        'justificativa': question_data.get('justificativa', 'ERRO'),
        'chunk1_id': id_similar,
        'chunk2_id': id_chunk_fora,
        'chunk1_text': chunk_similar,
        'chunk2_text': chunk_fora,
        'score_cosine': score_similar,
        'source_csv_row': row_index
    }


In [ ]:
CSV_ENTRADA = "resultado_similaridade.csv"
JSON_SAIDA = "perguntas_parciais_simil.json"
PROMPT_TEMPLATE = "prompts/composeN.txt"  
MODELO_LLM = "gpt-4o-mini" 
   
try:
    df = pd.read_csv(CSV_ENTRADA)
    print(f"CSV '{CSV_ENTRADA}' carregado. Processando {len(df)} linhas.")
except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo de entrada '{CSV_ENTRADA}' não encontrado.")
    df = pd.DataFrame()

if not df.empty:
    resultados_finais = []
    
    for row in df.itertuples():
        print(f"Processando linha {row.Index}...")
        resultado = gerar_pergunta_da_linha_csv(row, row.Index, PROMPT_TEMPLATE, MODELO_LLM)
        
        if resultado is not None:
            resultados_finais.append(resultado)
        
        time.sleep(1) # pausa para não sobrecarregar a API

    with open(JSON_SAIDA, 'w', encoding='utf-8') as f:
        json.dump(resultados_finais, f, indent=2, ensure_ascii=False)

    
    print(f"{len(resultados_finais)} perguntas foram geradas e salvas em '{JSON_SAIDA}'.")

In [ ]:
pd.DataFrame.from_records(resultados_finais).to_csv("25_parciais_gpt_simil.csv", index=False)